# Homework 5 — Learning Inverse Kinematics with Reinforcement Learning

### Purpose

One of the goals of this course is to:

> **Put the robot arm's end-effector at a commanded position *and* orientation.**

You have solved this with Newton's method on the Jacobian, and with operational space
control. Both need a model of the arm and its derivatives. This homework asks whether a
neural network can **learn** to do it instead, from data and from trial and error, and
what that costs.

The learned controller is a **policy**. Every 20 ms it reads the arm's joint angles and
the target pose, and moves the joint set-point. A joint PD controller with gravity and
Coriolis compensation, running underneath at 200 Hz, makes the arm follow the set-point
through real dynamics. Every episode lasts 0.6 s: the policy has to **reach** the target
and then **hold** it until the end.

**Overview.**

1. **Q1 Data** — generate validated target poses: random joint angles, forward
   kinematics, and collision checks against the floor and the arm itself.
2. **Q2 Reward** — train with three different rewards and find out why they behave so
   differently.
3. **Q3 Validated vs random targets** — train on targets drawn uniformly from the task box
   (some of which may not be reachable) and compare, on a fair test set.
4. **Q4 Imitation** — learn from (pose, joint angles) pairs, see where that fails and how to
   fix it, and whether it helps reinforcement learning.
5. **Q5 Comparison** — the best learned policy against numerical inverse kinematics.

**Grading:** 100 points.

**Time.** There are five training runs of 0.8 million steps, each roughly 5–8 minutes on a
laptop CPU, so plan about 45 minutes of computer time (longer training is optional, see the
setup cell). Start the runs early; you can write answers while they
train. **You must train every policy yourself.** Each run saves your policy in
`checkpoints/my_rl_<name>.pt`. If you later want to re-run the analysis without training
again, set `RETRAIN = False` in the setup cell: the training cells then reload *your own*
saved policies instead of training.

**Packages:** `mujoco`, `numpy`, `matplotlib`, `torch`, `gymnasium`
(`pip install torch gymnasium` if you do not have them).

### Conventions

* Pose errors are 6-vectors $[\,\omega\,;\,v\,]$: rotation error
  $\log(R_{des}R^{\top})^{\vee}$ first, position error $p_{des} - p$ second.
* **Task region.** Targets lie in a box in front of the robot, centered at
  $(0.6, 0, 0.35)$ m with half-width $0.3$ m, so from just above the floor
  ($z = 0.05$ m) to $0.65$ m. The tool orientation is limited to
  $|\text{roll}| \le 70^\circ$, $|\text{pitch}| \le 50^\circ$, $|\text{yaw}| \le 30^\circ$
  around the tool pointing straight forward.
* **Success** means position error below 20 mm **and** orientation error below 0.2 rad
  **at the end of the episode** (after 0.6 s), i.e. the target was reached and held.
* The package `forte_rl/` next to this notebook holds the given code: the arm and
  collision checker (`robot.py`), the task (`task.py`), the environment (`env.py`), the
  PPO trainer (`ppo.py`), numerical IK (`inverse_kinematics.py`), imitation-learning helpers (`imitation_learning.py`)
  and the evaluation harness (`rollout.py`). Read them; you will not need to change them.

In [ ]:
# No need to change
import os, time
import numpy as np
import matplotlib.pyplot as plt
import torch

from forte_rl.robot import Arm, CollisionChecker, pose_error, rot6d, rpy_from_R, R_from_rpy, JOINT_NAMES
from forte_rl.task import BOX_CENTER, BOX_HALF, RPY_LIMIT, POS_TOL, ORI_TOL, in_task_region, is_success
from forte_rl.env import EnvConfig, ForteReachEnv
from forte_rl.ppo import PPOConfig, train, load_checkpoint, init_from_checkpoint
from forte_rl.inverse_kinematics import DLSIK
from forte_rl.imitation_learning import fit_mlp, behavior_cloning
from forte_rl.rollout import PolicyController, IKController, evaluate

TRAIN_STEPS = 800_000         # environment steps per training run (about 5-7 min each).
                              # Optional: 1_000_000 or more learns further (better success and precision)
                              # if you have the time; keep the same value for every run you compare.
RETRAIN = True                # False: reload YOUR OWN policies saved by an earlier run of this notebook
torch.set_num_threads(4)
os.makedirs("data", exist_ok=True); os.makedirs("checkpoints", exist_ok=True)

arm = Arm()
checker = CollisionChecker()
ik = DLSIK(arm)

def train_or_load(name, reward, **kwargs):
    """Train a policy and save it as checkpoints/my_rl_<name>.pt, or, with RETRAIN = False,
    reload the policy YOU trained and saved under that name in an earlier run.
    kwargs are passed to train() (targets=..., target_sampler=..., eval_targets=..., ppo_cfg=..., init=...).
    Returns (policy, history)."""
    path = f"checkpoints/my_rl_{name}.pt"
    if not RETRAIN and os.path.exists(path):
        policy, _ = load_checkpoint(path)                                     # the best policy of that run
        history = load_checkpoint(path.replace(".pt", "_final.pt"))[1]["history"]
        print(f"(reloaded your own policy from {path})")
        return policy, history
    return train(reward, ckpt_path=path, **kwargs)

def check(name, got, want, tol=1e-8):
    try:
        ok = np.allclose(np.asarray(got, float), np.asarray(want, float), atol=tol)
    except (ValueError, TypeError):          # e.g. a list of shapes
        ok = got == want
    print(f'{"PASS" if ok else "FAIL"}  {name}')
    if not ok:
        print('   yours   :', got); print('   expected:', want)
    return ok

print("joint limits lo:", arm.q_lo.round(2), "\n             hi:", arm.q_hi.round(2))
print("ready_pose q   :", arm.q_home, "\ntool at ready_pose:", arm.fk(arm.q_home)[0].round(3))

In [ ]:
# Given: what ready_pose (the home pose, arm.q_home) looks like. Every episode starts here.
# Rendering is given and not needed for any answer. On Linux without a display, set MUJOCO_GL=egl.
import sys; sys.path.append('../ForteArm_Model')
from forte_viz import ForteViz, show_video, show_filmstrip
viz = ForteViz(width=1040, height=480)
home_pose = arm.fk(arm.q_home)                          # the RGB frame marks the tool pose (x red, y green, z blue)
views = [viz.camera(lookat=[0.3, 0.0, 0.35]),           # iso view
         viz.camera(azimuth=90, elevation=0, lookat=[0.3, 0.0, 0.35])]   # side view (x to the right)
for c in views:
    c.distance = 2.0
show_filmstrip([viz.frame(arm.q_home, home_pose, cam=c) for c in views], n=2, cols=2,
               title=f"ready_pose  q = {np.round(arm.q_home, 2)} rad")

---
# Q1 [15 pts] A dataset of valid target poses

A target is only worth training on if the arm can actually get there. The simplest way to
guarantee that is to **start from joint angles**: draw a random configuration $q$, compute
where the tool ends up, $(p, R) = \mathrm{FK}(q)$, and keep the pose if

1. it lies in the task region (`in_task_region(p, R)`), and
2. the arm in configuration $q$ touches neither the floor nor itself.

Every kept pose is then reachable by construction, and you also know one joint
configuration that reaches it.

### Tools you can use (already written for you)

You do **not** need to write forward kinematics or a collision checker. They are in
`forte_rl/robot.py` and `forte_rl/task.py`, and the setup cell already created the
objects `arm` and `checker`. Everything Q1 needs:

| call | what it gives you | defined in |
|---|---|---|
| `arm.q_lo`, `arm.q_hi` | lower / upper joint limits, arrays of shape (7,) [rad] | `robot.py`, class `Arm` |
| `arm.q_home` | the `ready_pose` joint angles (7,) | `robot.py`, class `Arm` |
| `p, R = arm.fk(q)` | forward kinematics: tool position (3,) [m] and rotation matrix (3, 3) | `robot.py`, `Arm.fk` |
| `checker.check(q)` | collision check, returns a dict `{'ground': bool, 'self': bool, 'pairs': [...]}` | `robot.py`, class `CollisionChecker` |
| `checker.in_collision(q)` | `True` if the arm touches the floor **or** itself | `robot.py`, class `CollisionChecker` |
| `in_task_region(p, R)` | `True` if $p$ is in the box **and** the orientation is within the limits | `task.py` |
| `rpy_from_R(R)` | roll, pitch, yaw (3,) [rad] of a rotation matrix | `robot.py` |
| `BOX_CENTER`, `BOX_HALF`, `RPY_LIMIT` | box center (3,) [m], half-width [m], roll/pitch/yaw limits (3,) [rad] | `task.py` |

About the collision checker: the arm model used for simulation has collisions switched off
(contact forces would disturb the dynamics). `CollisionChecker` keeps a private copy of
the model with collisions on, puts it in configuration $q$, and lists the contacts. The
collision shapes are the convex hulls of the link meshes, so the check is slightly
conservative. `'ground'` means some link touches the floor; `'self'` means two links of
the arm touch each other; `'pairs'` names the touching parts.

The next cell tries each tool once on `ready_pose`. Read it before you start.

**(a)** Complete `generate_valid_dataset(n, seed)`. Draw $q$ uniformly within the joint
limits (`arm.q_lo`, `arm.q_hi`) with `np.random.default_rng(seed)`, and keep drawing until
you have `n` valid poses. Count every
draw and the reason it was rejected in `stats`: `'draws'`, `'outside'` (not in the task
region), `'ground'`, `'self'`. Check the floor before self-collision, so a pose that hits
both counts as `'ground'`. Return a dict with arrays `q` $(n, 7)$, `p` $(n, 3)$,
`R` $(n, 3, 3)$, and `stats`.

**(b)** The cells after your function generate the training set (3,000 poses, about a
minute) and a test set (300 poses), show where the rejected poses are, how well the kept
poses cover the task region, and what rejected configurations look like. Answer in 4–6
sentences: What fraction of the in-region poses did the floor and self-collision checks
reject, and why is the floor a problem for this task region? Which parts of
the task region (positions and orientations) are covered densely and which sparsely?
Why is the coverage not uniform, even though $q$ was drawn uniformly?

In [ ]:
# Given: try the tools on ready_pose and on one random configuration
p, R = arm.fk(arm.q_home)
print("ready_pose: tool position", p.round(3), "  roll/pitch/yaw [deg]", np.degrees(rpy_from_R(R)).round(1))
print("   in the task region?", in_task_region(p, R))
print("   collisions:", checker.check(arm.q_home))

rng_demo = np.random.default_rng(0)
q = rng_demo.uniform(arm.q_lo, arm.q_hi)              # one random configuration within the joint limits
p, R = arm.fk(q)
print("random q  :", q.round(2))
print("   tool position", p.round(3), "  in the task region?", in_task_region(p, R))
print("   collisions:", checker.check(q))

# keep drawing until one configuration collides, to see what check() reports then
while True:
    q = rng_demo.uniform(arm.q_lo, arm.q_hi)
    hit = checker.check(q)
    if hit["ground"] or hit["self"]:
        break
print("colliding q:", q.round(2), "\n   collisions:", hit)
print("   in_collision(q):", checker.in_collision(q))

In [ ]:
def generate_valid_dataset(n, seed=0):
    """n reachable, collision-free target poses in the task region, with the q that reaches each."""
    rng = np.random.default_rng(seed)
    Q, P, R = [], [], []
    stats = {"draws": 0, "outside": 0, "ground": 0, "self": 0}
    # TODO Q1a: until you have n poses: draw q uniformly in [arm.q_lo, arm.q_hi] with rng,
    #   count the draw, compute (p, R) = arm.fk(q), reject (and count) if not in_task_region,
    #   then call checker.check(q) and reject on 'ground' first, then on 'self'.
    #   Append kept q, p, R to Q, P, R.
    raise NotImplementedError("Q1: generate_valid_dataset")
    return {"q": np.array(Q), "p": np.array(P), "R": np.array(R), "stats": stats}

In [ ]:
# --- self test on a small set ---
small = generate_valid_dataset(50, seed=123)
check('Q1 shapes', [small["q"].shape, small["p"].shape, small["R"].shape], [(50, 7), (50, 3), (50, 3, 3)])
check('Q1 every pose is in the task region', all(in_task_region(p, R) for p, R in zip(small["p"], small["R"])), True)
check('Q1 every q reproduces its pose', max(np.abs(arm.fk(q)[0] - p).max() for q, p in zip(small["q"], small["p"])), 0.0, tol=1e-9)
check('Q1 no stored configuration collides', any(checker.in_collision(q) for q in small["q"]), False)
st = small["stats"]
check('Q1 stats add up: draws = kept + outside + ground + self', st["draws"], 50 + st["outside"] + st["ground"] + st["self"])

In [ ]:
t0 = time.time()
train_set = generate_valid_dataset(3000, seed=0)
test_set = generate_valid_dataset(300, seed=1)
print(f"generated in {time.time() - t0:.0f} s")
st = train_set["stats"]; inreg = st["draws"] - st["outside"]
print(f"draws {st['draws']:,}: outside the region {st['outside']:,} ({100 * st['outside'] / st['draws']:.2f}%)")
print(f"of the {inreg:,} in-region poses: floor {st['ground']} ({100 * st['ground'] / inreg:.0f}%), "
      f"self {st['self']} ({100 * st['self'] / inreg:.0f}%), kept {3000} ({100 * 3000 / inreg:.0f}%)")
for name, s in (("train", train_set), ("test", test_set)):
    np.savez(f"data/valid_{name}.npz", q=s["q"], p=s["p"], R=s["R"])

### Where are the rejected poses, and how much of the task region is covered?

Given code. The first figure draws new random configurations and shows the tool position of
every in-region pose, colored by what happened to it: kept, rejected for the floor, or
rejected for self-collision. Two views: from the side ($x$–$z$) and from above ($x$–$y$).
The dashed rectangle is the task box. The second figure shows how the kept poses are
spread over the box and over the orientation limits (dashed lines).

In [ ]:
# classify fresh in-region poses (about 30 s)
rng = np.random.default_rng(7); cls = {"kept": [], "floor": [], "self": []}
while sum(len(v) for v in cls.values()) < 3000:
    q = rng.uniform(arm.q_lo, arm.q_hi); p, R = arm.fk(q)
    if not in_task_region(p, R):
        continue
    h = checker.check(q)
    cls["floor" if h["ground"] else "self" if h["self"] else "kept"].append(p)
cls = {k: np.array(v).reshape(-1, 3) for k, v in cls.items()}   # (n, 3), also when a class is empty
colors = {"kept": "tab:blue", "floor": "tab:red", "self": "tab:orange"}

lo, hi = BOX_CENTER - BOX_HALF, BOX_CENTER + BOX_HALF
fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
for a, (i, j, lab) in zip(ax, [(0, 2, "side view (x-z)"), (0, 1, "top view (x-y)")]):
    for k in ("kept", "floor", "self"):
        P = cls[k]
        a.scatter(P[:, i], P[:, j], s=4, c=colors[k], alpha=0.6, label=f"{k} ({len(P)})")
    a.add_patch(plt.Rectangle((lo[i], lo[j]), hi[i] - lo[i], hi[j] - lo[j], fill=False, ls="--", color="k"))
    a.set_xlabel("xyz"[i] + " [m]"); a.set_ylabel("xyz"[j] + " [m]"); a.set_title(lab)
    a.set_aspect("equal"); a.grid(alpha=0.3)
ax[0].legend(loc="upper right", markerscale=3)
plt.tight_layout(); plt.show()

# coverage of the kept training poses: position along each axis, and roll / pitch / yaw
P = train_set["p"]; RPY = np.degrees(np.array([rpy_from_R(R) for R in train_set["R"]]))
fig, ax = plt.subplots(1, 6, figsize=(16, 2.8))
for i in range(3):
    ax[i].hist(P[:, i], bins=25, range=(lo[i], hi[i]), color="tab:blue")
    ax[i].set_xlabel("xyz"[i] + " [m]")
for i, name in enumerate(("roll", "pitch", "yaw")):
    L = np.degrees(RPY_LIMIT[i])
    ax[3 + i].hist(RPY[:, i], bins=25, range=(-L, L), color="tab:green")
    ax[3 + i].axvline(-L, ls="--", c="k"); ax[3 + i].axvline(L, ls="--", c="k")
    ax[3 + i].set_xlabel(name + " [deg]")
ax[0].set_ylabel("number of poses")
fig.suptitle("coverage of the 3,000 training poses"); plt.tight_layout(); plt.show()

### What the collision check removes

Rejected configurations, rendered. (Rendering is *given* and not needed for any answer.
On Linux without a display, set `MUJOCO_GL=egl`.)

In [ ]:
# uses the renderer `viz` created in the ready_pose cell at the top
cam = viz.camera(lookat=[0.45, 0.0, 0.3]); cam.distance = 1.7     # close-up on the task region
rng = np.random.default_rng(11); shown = []
while len(shown) < 4:                       # four in-region poses rejected for touching the floor
    q = rng.uniform(arm.q_lo, arm.q_hi); p, R = arm.fk(q)
    if in_task_region(p, R) and checker.check(q)["ground"]:
        shown.append(q)
frames = [viz.frame(q, arm.fk(q), cam=cam) for q in shown]
show_filmstrip(frames, n=4, cols=4, title="rejected for touching the floor, tool in the task region (RGB frame = the tool pose)")

**(Your answer for Q1b)**

*write here*

---
# Q2 [25 pts] Which reward?

**The environment** (`forte_rl/env.py`). One step is one decision, 20 ms long.

* **Observation** (53 numbers): $\cos q$, $\sin q$, $0.1\,\dot q$, the set-point offset
  $q_{cmd} - q$, the tool pose, the target pose (positions plus the first two columns of
  each rotation matrix), the pose error $[\omega; v]$, and the fraction of the episode
  used.
* **Action** $a \in [-1, 1]^7$ moves the joint set-point:
  $q_{cmd} \leftarrow q_{cmd} + 0.2\,a$ rad, kept inside the joint limits and within
  0.5 rad of the actual $q$.
* **Underneath**, at 200 Hz: $\tau = K_p(q_{cmd} - q) - K_d\dot q + (C + G)$, clipped to
  the motor limits.
* **Episodes** start at `ready_pose` and always last 30 steps (0.6 s). They do **not** end
  when the target is reached: the policy must reach it and then stay there. **Success**
  means the tool is inside the tolerance at the last step.

The reward is the only thing that tells the learner what we want, so its shape matters.
Measure both errors **in units of their tolerance**,
$e_p = \|p_{err}\| / \texttt{POS\_TOL}$ and $e_o = \|o_{err}\| / \texttt{ORI\_TOL}$, so that
"1" means "one tolerance away" for both. Three candidates, each minus a small action
penalty $0.01\,\|a\|^2$ (it keeps the set-point from thrashing):

| name | error terms |
|---|---|
| `reward_linear` | $-0.05\,(e_p + e_o)$ |
| `reward_inverse` | $\dfrac{1}{1 + e_p} + \dfrac{1}{1 + e_o}$ |
| `reward_combined` | $-0.05\,(e_p + e_o) + \dfrac{1}{1 + e_p} + \dfrac{1}{1 + e_o}$ |

(Why $1/(1+e)$ and not $1/e$? $1/e$ goes to infinity at zero error, so a single lucky step
would dominate everything the learner sees. $1/(1+e)$ has the same shape but stays
between 0 and 1.)

**(a)** Complete the three functions. Each is called after every step with the position
error (3,) [m], the orientation error (3,) [rad] and the action (7,), and returns a float.

**(b)** Train one policy per reward on the validated training targets, with the same budget
(`TRAIN_STEPS`). Use
`train_or_load(name, reward, targets=(p, R), eval_targets=eval_T, ppo_cfg=PPOConfig(total_timesteps=TRAIN_STEPS))`
(given in the setup cell; it calls `train` and saves the result) and store each returned
`(policy, history)` pair in `runs["linear"]`, `runs["inverse"]`,
`runs["combined"]`. Each run takes about 5–10 minutes and prints its progress. Every few
updates the policy is tested on the first 100 validated test targets.

**(c)** *Optional:* design your own reward in `reward_mine`, train it the same way, and add it
to the comparison.

**(d)** Answer in 6–10 sentences. Which reward works best, and how do they differ in
success, position error and orientation error? Explain each result from the *shape* of
the reward: how strongly does each one push when the tool is far away, and when it is
already close? What does a policy trained on `reward_inverse` do, and why is that a
sensible thing to do *for that reward*? (Reinforcement learning is noisy: another random
seed, `PPOConfig(seed=1)`, can give a different result. If you have time, repeat a run.)

In [ ]:
def reward_linear(pos_err, ori_err, action):
    """-error."""
    # TODO Q2a: ep = |pos_err| / POS_TOL, eo = |ori_err| / ORI_TOL;
    #   return -0.05 (ep + eo) - 0.01 sum(action**2) as a float.
    raise NotImplementedError("Q2: reward_linear")


def reward_inverse(pos_err, ori_err, action):
    """Bounded 1/error."""
    # TODO Q2a: return 1/(1 + ep) + 1/(1 + eo) - 0.01 sum(action**2) as a float.
    raise NotImplementedError("Q2: reward_inverse")


def reward_combined(pos_err, ori_err, action):
    """-error + bounded 1/error."""
    # TODO Q2a: return -0.05 (ep + eo) + 1/(1 + ep) + 1/(1 + eo) - 0.01 sum(action**2) as a float.
    raise NotImplementedError("Q2: reward_combined")


In [ ]:
# --- self test ---
z, a0 = np.zeros(3), np.zeros(7)
P = lambda d: np.array([d, 0.0, 0.0])                # a position error of d meters
O = lambda d: np.array([d, 0.0, 0.0])                # an orientation error of d radians
check('Q2 linear: one tolerance away in both', reward_linear(P(POS_TOL), O(ORI_TOL), a0), -0.1, tol=1e-6)
check('Q2 inverse: zero error', reward_inverse(z, z, a0), 2.0, tol=1e-6)
check('Q2 inverse: one tolerance away in both', reward_inverse(P(POS_TOL), O(ORI_TOL), a0), 1.0, tol=1e-6)
check('Q2 combined: one tolerance away in both', reward_combined(P(POS_TOL), O(ORI_TOL), a0), 0.9, tol=1e-6)
check('Q2 combined = linear + inverse', reward_combined(P(0.13), O(0.4), a0), reward_linear(P(0.13), O(0.4), a0) + reward_inverse(P(0.13), O(0.4), a0), tol=1e-6)
check('Q2 action penalty (all three)', [r(z, z, np.ones(7)) - r(z, z, a0) for r in (reward_linear, reward_inverse, reward_combined)], [-0.07] * 3, tol=1e-6)

# How much does each reward change between two poses? (no action penalty)
print("\nreward gained by moving closer:          linear  inverse  combined")
for far, near in ((0.40, 0.30), (0.10, 0.05), (0.018, 0.002)):
    g = [r(P(near), z, a0) - r(P(far), z, a0) for r in (reward_linear, reward_inverse, reward_combined)]
    print(f"   position error {1000 * far:5.0f} mm -> {1000 * near:4.0f} mm:   " + "  ".join(f"{x:+7.3f}" for x in g))

# Always check that the task can be solved before training on it: numerical IK + the same PD.
env = ForteReachEnv(reward_combined, targets=(test_set["p"], test_set["R"]))
r_ik = evaluate(IKController(ik, restarts=3), env, (test_set["p"][:100], test_set["R"][:100]))
print(f"\nIK + PD on 100 test targets: {100 * r_ik['success'].mean():.0f}% inside the tolerance at the end, "
      f"first reached after {np.median(r_ik['steps']):.0f} steps (median; the episode has 30)")

In [ ]:
eval_T = (test_set["p"], test_set["R"])
runs = {}

# TODO Q2b: for each (name, reward) in (('linear', reward_linear), ('inverse', reward_inverse),
#   ('combined', reward_combined)): runs[name] = train_or_load(name, reward, targets=(train_set['p'], train_set['R']),
#   eval_targets=eval_T, ppo_cfg=PPOConfig(total_timesteps=TRAIN_STEPS))
raise NotImplementedError("Q2: training runs")


In [ ]:
def plot_curves(names, title, ylabel="held at 0.6 s, 100 validated test targets [%]"):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    for n in names:
        h = runs[n][1]; x = np.array(h["step"]) / 1e6
        ax[0].plot(x, 100 * np.array(h["success"]), "o-", ms=3, label=n)
        ax[1].plot(x, 1000 * np.array(h["pos_med"]), "o-", ms=3, label=n)
        ax[2].plot(x, np.array(h["ori_med"]), "o-", ms=3, label=n)
    ax[0].set_ylabel(ylabel); ax[1].set_ylabel("final median position error [mm]")
    ax[2].set_ylabel("final median orientation error [rad]")
    ax[1].set_yscale("log"); ax[2].set_yscale("log")
    for a in ax:
        a.set_xlabel("environment steps [millions]"); a.grid(alpha=0.3); a.legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()
    for n in names:
        h = runs[n][1]
        print(f"{n:10s}: final success {100 * h['success'][-1]:5.1f}%, pos med {1000 * h['pos_med'][-1]:6.1f} mm, "
              f"ori med {h['ori_med'][-1]:.3f} rad")

plot_curves(["linear", "inverse", "combined"], "Q2: three rewards, validated targets")

In [ ]:
# (c) optional: your own reward
def reward_mine(pos_err, ori_err, action):
    ep, eo = np.linalg.norm(pos_err) / POS_TOL, np.linalg.norm(ori_err) / ORI_TOL
    return reward_linear(pos_err, ori_err, action)          # change me: your own reward from ep, eo, action

RUN_MY_REWARD = False
if RUN_MY_REWARD:
    runs["mine"] = train_or_load("mine", reward_mine, targets=(train_set["p"], train_set["R"]), eval_targets=eval_T,
                                 ppo_cfg=PPOConfig(total_timesteps=TRAIN_STEPS))
    plot_curves(["linear", "inverse", "combined", "mine"], "Q2: with your own reward")

reward_fn = reward_combined          # used for the rest of the homework

**(Your answer for Q2d)**

*write here*

---
# Q3 [15 pts] Validated targets versus random box targets

Q1 built the training targets carefully: every target comes from a joint configuration
(forward kinematics) that passed the collision check, so we know the arm can reach it.
Another way to sample targets is to pick the target pose directly from the ranges we
specified: a position **uniformly in the task box**, and roll, pitch and yaw each
**uniformly within their limits**, without checking whether the arm can reach it. Such a
target is always inside the task region, but some of these targets cannot be reached at
all. This question explores how the way the targets are sampled affects the performance
of the final policy.

To compare fairly, both policies are tested on **two** test sets: the validated test set
from Q1, and a new set of 300 **randomly sampled box targets**, which is what a user of the
robot might actually ask for.

**(a)** Complete `sample_box_target(rng)`. Use `BOX_CENTER`, `BOX_HALF`, `RPY_LIMIT`, and
`R_from_rpy([roll, pitch, yaw])` from `robot.py`, which returns
$R = R_z(\text{yaw})\,R_y(\text{pitch})\,R_x(\text{roll})$.

**(b)** Train `runs["box"] = train_or_load("box", reward_fn, ...)` with a fresh box target
every episode (`target_sampler=sample_box_target`), the same budget, and the box test set
as `eval_targets` (`eval_box`).

**(c)** The next cell plots the learning curves of both policies, `runs["combined"]` from Q2
(trained on validated targets) and `runs["box"]`, each tested on **both** test sets: the
policy saved at every evaluation during training is tested again on the first 100 targets
of each set (about a minute). It then tests the two trained policies on all 300 targets of
both sets.

**(d)** Answer in 5–8 sentences. Which policy is better on the validated test set, and on
the box test set? Is that what you expected? Some random box targets cannot be reached at
all: what does the reward ask for on such a target, and what does that do to the value
function? On the other hand, what do random box targets give the learner that the
validated set does not (look back at the Q1 coverage plot)? Why is the success rate on the
box test set lower for *every* policy, and what would be the best possible score there?

In [ ]:
def sample_box_target(rng):
    """A target pose drawn uniformly from the task box and orientation limits (not checked)."""
    # TODO Q3a: p = BOX_CENTER + uniform(-BOX_HALF, BOX_HALF) in each axis;
    #   roll, pitch, yaw uniform in [-RPY_LIMIT, RPY_LIMIT]; R = R_from_rpy(rpy).
    raise NotImplementedError("Q3: sample_box_target")
    return p, R


def generate_box_targets(n, seed=0):
    rng = np.random.default_rng(seed)
    pr = [sample_box_target(rng) for _ in range(n)]
    return {"p": np.array([a for a, _ in pr]), "R": np.array([b for _, b in pr])}

In [ ]:
# --- self test ---
bx = generate_box_targets(500, seed=7)
check('Q3 box positions inside the box', np.all(np.abs(bx["p"] - BOX_CENTER) <= BOX_HALF), True)
check('Q3 box orientations inside the limits', all(np.all(np.abs(rpy_from_R(R)) <= RPY_LIMIT + 1e-9) for R in bx["R"]), True)
check('Q3 box positions spread uniformly (mean near the center)', np.abs(bx["p"].mean(0) - BOX_CENTER).max() < 0.03, True)
check('Q3 box orientations spread uniformly (mean near zero)', np.abs(np.mean([rpy_from_R(R) for R in bx["R"]], 0)).max() < 0.1, True)

box_set = generate_box_targets(300, seed=2)          # the random box test set
np.savez("data/box_test.npz", p=box_set["p"], R=box_set["R"])
eval_box = (box_set["p"], box_set["R"])

In [ ]:
# TODO Q3b: runs['box'] = train_or_load('box', reward_fn, target_sampler=sample_box_target,
#   eval_targets=eval_box, ppo_cfg=PPOConfig(total_timesteps=TRAIN_STEPS))
raise NotImplementedError("Q3: training run")


In [ ]:
# learning curves of both policies, each tested on BOTH test sets: the policy saved at every
# evaluation during training (saved in checkpoints/my_rl_<name>/) is tested again on the first
# 100 targets of each set
from forte_rl.ppo import load_snapshots, quick_eval
tests = {"validated test set": eval_T, "box test set": eval_box}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for name, label in (("combined", "trained on validated targets"), ("box", "trained on random box targets")):
    snaps = load_snapshots(f"checkpoints/my_rl_{name}")          # checkpoints/my_rl_<name>/step_*.pt
    x = [step / 1e6 for step, _ in snaps]
    for a, T in zip(ax, tests.values()):
        y = [100 * quick_eval(pol, reward_fn, T, EnvConfig(), 100)["success"] for _, pol in snaps]
        a.plot(x, y, "o-", ms=3, label=label)
for a, tname in zip(ax, tests):
    a.set_title(f"tested on the {tname}"); a.set_xlabel("environment steps [millions]")
    a.grid(alpha=0.3); a.legend()
ax[0].set_ylabel("held at 0.6 s, first 100 test targets [%]")
fig.suptitle("Q3: both policies on both test sets during training"); plt.tight_layout(); plt.show()

# both policies on both test sets (all 300 targets of each)
env_v = ForteReachEnv(reward_fn, targets=eval_T)
print(f'\n{"":28s}{"validated test set":>30s}{"box test set":>30s}')
print(f'{"trained on":28s}' + 2 * f'{"success":>10s}{"pos med":>10s}{"ori med":>10s}')
for name in ("combined", "box"):
    line = f'{"validated targets" if name == "combined" else "random box targets":28s}'
    for T in (eval_T, eval_box):
        r = evaluate(PolicyController(runs[name][0]), env_v, T)
        line += f'{100 * r["success"].mean():9.1f}%{1000 * np.median(r["pos_err"]):8.1f}mm{np.median(r["ori_err"]):9.3f}r'
    print(line)

**(Your answer for Q3d)**

*write here*

---
# Q4 [25 pts] Imitation learning: does it help?

The dataset already contains (pose, joint angles) pairs. Can we learn inverse kinematics
directly from them by supervised regression, and use that to start reinforcement learning?

**(a) Naive regression.** Train a network to map a target pose to joint angles using the
raw pairs: input $x = [p, \text{first two columns of } R]$ (9 numbers, use
`np.hstack([P, rot6d(R)])`), output $y = q$ (7 numbers). Use `fit_mlp(X, Y)` (given,
mean-squared error). The cell then measures where the predicted joint angles actually put
the tool. The cell before it counts how many distinct exact solutions a pose has.

**(b) Consistent labels.** Relabel every training pose with the solution numerical IK
finds **from `ready_pose`** (`ik.solve(arm.q_home, p, R)` returns `(q, converged, iters)`).
Keep only the poses where it converged, and store their indices in `keep` and the new
labels in `Q_relabel`. Fit and evaluate again.

**(c) Behavior cloning.** Now imitate a *controller* instead of a single answer. The expert
starts at `ready_pose` and moves the set-point toward its label at the maximum rate:
$a = \mathrm{clip}\big((q_{label} - q_{cmd}) / 0.2,\ -1,\ 1\big)$, using `env.q_cmd`.
For every relabeled training pose, reset the environment with
`env_bc.reset(seed=j, options={"target_index": j})`, roll the expert out for the whole
episode (30 steps: once the set-point reaches the label, the action becomes zero and the
expert *holds*), and record each observation the expert saw and the action it took, in `O`
and `A`. `behavior_cloning(O, A)` (given) then trains the policy network on them.

**(d)** The next cells evaluate the cloned policy, then continue training it with RL
(`init=` in `train`) using `reward_fn`. Fine-tuning uses a **much smaller learning rate**
($5\times10^{-5}$ instead of the default $3\times10^{-4}$) and trains the critic alone for
the first 20 updates.

**(e)** Answer these:

1. Why does naive regression fail, even on its own training data?
2. Relabeling helps a lot but still rarely reaches the 20 mm tolerance. Why is predicting
   the answer in one shot so much harder than the closed-loop task in (c)?
3. Compare RL from scratch (`runs["combined"]`), behavior cloning alone, and behavior
   cloning followed by RL. Did imitation help, and with what: the final result, the
   speed of learning, the reliability?
4. Why does fine-tuning a cloned policy use a smaller learning rate and train the critic
   alone first? What can go wrong when a good starting policy meets a randomly
   initialized critic?

In [ ]:
X_test = np.hstack([test_set["p"], rot6d(test_set["R"])])

def fk_errors(Q_pred, P, R):
    """Where do predicted joint angles put the tool? Returns (pos errors, ori errors, success rate)."""
    pe, oe = [], []
    for q, p, Rd in zip(Q_pred, P, R):
        pp, RR = arm.fk(np.clip(q, arm.q_lo, arm.q_hi))
        e = pose_error(pp, RR, p, Rd); pe.append(np.linalg.norm(e[3:])); oe.append(np.linalg.norm(e[:3]))
    pe, oe = np.array(pe), np.array(oe)
    return pe, oe, np.mean([is_success(a, b) for a, b in zip(pe, oe)])

# How many exact solutions does one pose have? Newton from 20 random starts, 40 test poses.
rng = np.random.default_rng(3); n_sol = []
for i in range(40):
    sols = []
    for _ in range(20):
        q, ok, _ = ik.solve(rng.uniform(arm.q_lo, arm.q_hi), test_set["p"][i], test_set["R"][i])
        if ok and not any(np.max(np.abs(q - s)) < 0.2 for s in sols):
            sols.append(q)
    n_sol.append(len(sols))
print(f"distinct solutions per pose: mean {np.mean(n_sol):.1f}, min {min(n_sol)}, max {max(n_sol)}; "
      f"{100 * np.mean(np.array(n_sol) >= 2):.0f}% of poses have two or more")

In [ ]:
# (a) naive regression on the raw pairs
# TODO Q4a: X_train = [p, rot6d(R)] of the training set (np.hstack), Y_train = its q,
#   naive, _ = fit_mlp(X_train, Y_train, epochs=300, verbose=False)
raise NotImplementedError("Q4: naive regression")
for name, X, S in (("test", X_test, test_set), ("its own training poses", X_train, train_set)):
    pe, oe, s = fk_errors(naive(X), S["p"], S["R"])
    print(f"naive, {name:22s}: pos med {1000 * np.median(pe):6.1f} mm, ori med {np.median(oe):.3f} rad, success {100 * s:.1f}%")

In [ ]:
# (b) consistent labels: the solution IK reaches from ready_pose
# TODO Q4b: for every training pose i: q, ok, _ = ik.solve(arm.q_home, p_i, R_i);
#   if ok, append i to keep and q to Q_relabel. Turn both into numpy arrays.
raise NotImplementedError("Q4: relabel")
print(f"IK from ready_pose converged on {len(keep)} / {len(train_set['p'])} training poses")
relabeled, _ = fit_mlp(X_train[keep], Q_relabel, epochs=300, verbose=False)
pe, oe, s = fk_errors(relabeled(X_test), test_set["p"], test_set["R"])
print(f"relabeled, test: pos med {1000 * np.median(pe):6.1f} mm, ori med {np.median(oe):.3f} rad, success {100 * s:.1f}%")
check('Q4 labels really solve their poses', max(np.linalg.norm(arm.fk(q)[0] - train_set["p"][i]) for q, i in zip(Q_relabel[:50], keep[:50])) < 1e-3, True)

In [ ]:
# (c) demonstrations of the closed-loop expert, then behavior cloning
env_bc = ForteReachEnv(reward_fn, targets=(train_set["p"][keep], train_set["R"][keep]))
O, A = [], []
# TODO Q4c: for j in range(len(keep)): obs, _ = env_bc.reset(seed=j, options={'target_index': j});
#   then up to env_bc.cfg.max_steps times: a = clip((Q_relabel[j] - env_bc.q_cmd) / env_bc.cfg.max_dq, -1, 1),
#   append obs to O and a to A, step the environment, stop when terminated or truncated.
raise NotImplementedError("Q4: demonstrations")
O, A = np.array(O), np.array(A)
print(f"{len(O):,} (observation, action) pairs from {len(keep)} demonstrations")
check('Q4 one action per observation', [O.shape[0], A.shape[1]], [A.shape[0], 7])
t0 = time.time()
bc_policy, bc_init, _ = behavior_cloning(O, A, epochs=50, verbose=False)
print(f"behavior cloning: {time.time() - t0:.0f} s")

In [ ]:
# (d) the cloned policy alone, then continued with RL
r_bc = evaluate(PolicyController(bc_policy), env_v, (eval_T[0][:100], eval_T[1][:100]))
print(f"behavior cloning alone: success {100 * r_bc['success'].mean():.0f}%, "
      f"pos med {1000 * np.median(r_bc['pos_err']):.1f} mm, ori med {np.median(r_bc['ori_err']):.3f} rad")

runs["bc_rl"] = train_or_load("bc_rl", reward_fn, targets=(train_set["p"], train_set["R"]), eval_targets=eval_T,
                              ppo_cfg=PPOConfig(total_timesteps=TRAIN_STEPS, learning_rate=5e-5, critic_warmup=20),
                              init=bc_init)
plot_curves(["combined", "bc_rl"], "Q4: RL from scratch vs behavior cloning + RL")

**(Your answer for Q4e)**

*write here*

---
# Q5 [20 pts] Learned policy versus numerical inverse kinematics

Now compare your best policy with the method you already know: **numerical IK** computes a
joint solution once at the start of the episode, and the set-point is then moved straight
toward it at the same maximum rate the policy has (0.2 rad per step). Both run in the same
environment with the same joint PD, so differences come only from how the joint targets
are chosen. IK starts from `ready_pose` and, if it does not converge, retries from random configurations (three attempts in total, "IK x3").

`evaluate` (given) returns per-target arrays, among them `success`, `pos_err`, `ori_err`
(at the end of the episode) and `steps` (decisions until the tolerance was first reached;
30 if never).

**(a)** Complete `summarize(res)`: return a dict with the success rate [%], the median
position error [mm], the median orientation error [rad], and the mean number of steps to
first reach the target **over successful episodes only**. The cell prints the table for the
validated and the box test sets.

**(b)** Answer in 6–10 sentences, using your numbers: Where does each method win (success,
precision, speed)? Why is IK so much more precise at the end, and why can the policy be
faster? What happens on the box test set, where some targets cannot be reached at all?
Which would you put on a real robot, and could you combine the two?

In [ ]:
def summarize(res):
    """Summary statistics of an `evaluate` result."""
    # TODO Q5a: s = res['success'] > 0.5. Return a dict with keys (in this order)
    #   'success %', 'pos med [mm]', 'ori med [rad]', 'steps' (mean over successful episodes).
    raise NotImplementedError("Q5: summarize")

def print_table(rows):
    keys = list(next(iter(rows.values())).keys())
    print(f'{"":26s}' + "".join(f"{k:>15s}" for k in keys))
    for name, d in rows.items():
        print(f"{name:26s}" + "".join(f"{d[k]:15.3f}" for k in keys))

best = max(("combined", "bc_rl"), key=lambda n: runs[n][1]["success"][-1])
print("best policy:", best)
best_policy = runs[best][0]
res = {}
for tname, T in (("validated", eval_T), ("box", eval_box)):
    res[f"RL policy | {tname}"] = evaluate(PolicyController(best_policy), env_v, T)
    res[f"IK x3 + PD | {tname}"] = evaluate(IKController(ik, restarts=3), env_v, T)
print_table({k: summarize(v) for k, v in res.items()})

### Watch them

Five validated targets, solved by the learned policy and by IK + PD. The yellow ball is the
tool; the RGB frame is the target. In the video the policy is on top, IK + PD below.

In [ ]:
pick = [0, 1, 2, 10, 20]
sel = (eval_T[0][pick], eval_T[1][pick])
trajs = {}
for name, ctl in [("RL policy", PolicyController(best_policy)), ("IK x3 + PD", IKController(ik, restarts=3))]:
    trajs[name] = evaluate(ctl, env_v, sel, record=True)["traj"]

# final configuration of each episode: top row RL policy, bottom row IK + PD
cam5 = viz.camera(lookat=[0.4, 0.0, 0.4], azimuth=140, elevation=-15); cam5.distance = 1.3   # close-up
finals = [viz.frame(trajs[name][j][-1], (sel[0][j], sel[1][j]), cam=cam5)
          for name in trajs for j in range(len(pick))]
show_filmstrip(finals, n=len(finals), cols=len(pick), title="final pose: RL policy (top) vs IK x3 + PD (bottom)")

In [ ]:
# video: RL policy (top) and IK x3 + PD (bottom), the same targets one after another
from PIL import Image, ImageDraw, ImageFont
try:
    font = ImageFont.load_default(size=30)
except TypeError:                                     # older Pillow: fixed small font
    font = ImageFont.load_default()

def labeled(img, text):
    im = Image.fromarray(img); ImageDraw.Draw(im).text((20, 15), text, fill=(255, 255, 255), font=font)
    return np.asarray(im)

video = []
for j in range(len(pick)):
    a, b = trajs["RL policy"][j], trajs["IK x3 + PD"][j]
    tgt = (sel[0][j], sel[1][j])
    for k in range(max(len(a), len(b)) + 10):         # hold the end a moment
        fa = viz.frame(a[min(k, len(a) - 1)], tgt, cam=cam5)
        fb = viz.frame(b[min(k, len(b) - 1)], tgt, cam=cam5)
        video.append(np.concatenate([labeled(fa, f"RL policy  (target {j + 1})"),
                                     labeled(fb, f"IK x3 + PD  (target {j + 1})")], axis=0))
show_video(video, fps=10, path="hw5_rl_vs_ik.mp4", width=1200)

**(Your answer for Q5b)**

*write here*

---
## Submission
Run all cells top to bottom — every `check(...)` should print **PASS** — then export to
PDF and upload it. Written answers are required for **Q1b, Q2d, Q3d, Q4e** and **Q5b**.